# 03 — Bias Classifier Training

Fine-tune RoBERTa-base on combined bias datasets for multi-class bias classification.

**Target:** ≥75% macro F1 across all 9 classes (8 biases + no_bias).

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
import mlflow

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Load Combined Dataset

In [ ]:
# Load your combined dataset
# This should be built from BBQ + CrowS-Pairs + WinoBias + your sycophancy dataset
# Each example should have: 'text' (reasoning trace) and 'label' (bias type index)

LABEL_MAP = {
    'confirmation_bias': 0,
    'anchoring_bias': 1,
    'availability_heuristic': 2,
    'sycophancy_bias': 3,
    'overconfidence_bias': 4,
    'framing_effect': 5,
    'recency_bias': 6,
    'bandwagon_effect': 7,
    'no_bias': 8,
}
ID2LABEL = {v: k for k, v in LABEL_MAP.items()}
NUM_LABELS = len(LABEL_MAP)

# TODO: Replace with your actual combined dataset loading
# with open('../data/combined_bias_dataset.json', 'r') as f:
#     data = json.load(f)
# df = pd.DataFrame(data)

# Placeholder: Create synthetic data for testing the pipeline
np.random.seed(42)
n_samples = 5000
texts = [f'Sample reasoning trace {i} with potential bias indicators.' for i in range(n_samples)]
labels = np.random.randint(0, NUM_LABELS, n_samples)
df = pd.DataFrame({'text': texts, 'label': labels})

print(f'Dataset size: {len(df)}')
print(f'Label distribution:\n{df["label"].value_counts().sort_index()}')

## 2. Tokenize & Split

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

# Split 80-10-10
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

def tokenize_fn(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=512)

train_ds = Dataset.from_pandas(train_df[['text', 'label']]).map(tokenize_fn, batched=True)
val_ds = Dataset.from_pandas(val_df[['text', 'label']]).map(tokenize_fn, batched=True)
test_ds = Dataset.from_pandas(test_df[['text', 'label']]).map(tokenize_fn, batched=True)

train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print('Tokenization complete.')

## 3. Train RoBERTa Classifier

In [ ]:
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL_MAP,
)

# Handle class imbalance with weighted loss
class_counts = train_df['label'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * NUM_LABELS
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print(f'Class weights: {class_weights.round(3)}')

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'f1_macro': f1, 'accuracy': acc}

training_args = TrainingArguments(
    output_dir='../ml_models/bias_classifier_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=50,
    report_to='none',  # Set to 'mlflow' to log to MLflow
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print('Starting training...')
trainer.train()

## 4. Evaluate on Test Set

In [ ]:
# Evaluate
predictions = trainer.predict(test_ds)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Classification report
target_names = [ID2LABEL[i] for i in range(NUM_LABELS)]
report = classification_report(labels, preds, target_names=target_names)
print(report)

# Confusion matrix
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_title('Bias Classifier — Confusion Matrix', fontsize=14)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Save Model

In [ ]:
save_path = '../ml_models/bias_classifier'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f'Model saved to {save_path}')
print('\nNext: Run notebook 04 for calibration experiments.')